# Allreduce -- The TP Decode Hot Path

Allreduce is the **single most latency-critical collective for inference**. In a TP=4 decode loop,
it fires ~2 times per transformer layer -- once after the attention output projection, once after the FFN down projection.
For an 80-layer model (e.g. Llama-3 70B) that is **~160 allreduces per generated token**, all on the critical path.

This notebook walks through:
1. Initializing oneCCL with the correct backend and environment
2. Running a basic allreduce and verifying correctness
3. Benchmarking Ring vs default selection on NUMA-only hardware
4. Simulating the TP decode allreduce pattern at realistic message sizes
5. Profiling with Intel VTune markers

:::{note}
All cells use a **launcher pattern**: the MPI workload is written to a temp script and
executed via `mpirun` as a subprocess. This lets you run the notebook from a single-rank
Jupyter kernel while still exercising real multi-rank collectives.
:::

## 0. Environment Check

In [ ]:
import subprocess, sys, shutil

def check(label, cmd, expect_in=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0 and (expect_in is None or expect_in in result.stdout + result.stderr)
    status = '✅' if ok else '❌'
    print(f"{status}  {label}")
    if not ok:
        print(f"     stdout: {result.stdout.strip()[:200]}")
        print(f"     stderr: {result.stderr.strip()[:200]}")
    return ok

check("mpirun available",         "which mpirun")
check("Intel MPI loaded",         "mpirun --version", expect_in="Intel")
check("PyTorch installed",        "python -c 'import torch; print(torch.__version__)'")
check("oneCCL bindings installed","python -c 'import oneccl_bindings_for_pytorch'")
check("XPU available",            "python -c 'import torch; assert torch.xpu.is_available()'")
check("NUMA tools available",     "which numactl")

In [ ]:
# Show GPU-to-NUMA mapping -- critical for understanding ring construction
import subprocess
result = subprocess.run("numactl --hardware", shell=True, capture_output=True, text=True)
print(result.stdout)

# Check XPU device count
result2 = subprocess.run(
    "python -c 'import torch; print(f\"XPU device count: {torch.xpu.device_count()}\")'" ,
    shell=True, capture_output=True, text=True
)
print(result2.stdout)

## 1. Basic Allreduce -- Correctness Verification

Each rank starts with a tensor filled with `rank + 1`. After allreduce (sum),
all ranks should hold `sum(1..N) = N*(N+1)/2`.

For N=4: expected value = **10** on every rank.

In [ ]:
%%writefile /tmp/ccl_allreduce_basic.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch  # registers 'ccl' backend

# --- Init ---
# oneCCL reads rank/size from MPI environment (PMI_RANK, MPI_LOCALRANKID, etc.)
os.environ.setdefault("CCL_ATL_TRANSPORT", "ofi")       # prefer OFI over MPI transport
os.environ.setdefault("CCL_WORKER_COUNT", "1")           # 1 worker thread for decode
os.environ.setdefault("CCL_LOG_LEVEL", "warn")
# NOTE: We set CCL_ALLREDUCE=ring here to benchmark the ring CPU algorithm explicitly.
# For production GPU inference, leave CCL_ALLREDUCE unset (defaults to "topo" which
# keeps data on-GPU). Setting it to "ring" forces a GPU→CPU→GPU copy path.
os.environ.setdefault("CCL_ALLREDUCE", "ring")

dist.init_process_group(backend="ccl")

rank = dist.get_rank()
world_size = dist.get_world_size()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# --- Create tensor ---
# Simulate a TP partial activation: (batch=1, seq=1, hidden=8192) shard
hidden_dim = 8192
tensor = torch.full((hidden_dim,), fill_value=float(rank + 1),
                    dtype=torch.bfloat16, device=device)

if rank == 0:
    print(f"[rank {rank}] Before allreduce: {tensor[:4].tolist()} ... (shape={tensor.shape})")

# --- Allreduce ---
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
torch.xpu.synchronize(device)

# --- Verify ---
expected = float(world_size * (world_size + 1) / 2)
actual = tensor[0].item()
correct = abs(actual - expected) < 0.1  # allow small BF16 rounding

print(f"[rank {rank}] After allreduce: {tensor[:4].tolist()}  "
      f"| expected={expected:.1f} | {'PASS' if correct else 'FAIL'}")

dist.destroy_process_group()

In [ ]:
# Launch with 4 ranks (adjust -n to match your device count)
import subprocess
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allreduce_basic.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 2. Async Allreduce -- Overlap Pattern

In inference the compute-comm overlap opportunity is limited (see [topology chapter](../chapters/02_topology)),
but `all_reduce` supports `async_op=True` which returns a Work handle.
You can issue the collective and do other bookkeeping before waiting.

This is useful for overlapping **KV cache management** with the allreduce on the TP critical path.

In [ ]:
%%writefile /tmp/ccl_allreduce_async.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"
os.environ["CCL_LOG_LEVEL"]     = "warn"

dist.init_process_group(backend="ccl")
rank  = dist.get_rank()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

hidden_dim = 8192
tensor = torch.ones(hidden_dim, dtype=torch.bfloat16, device=device) * (rank + 1)

# Fire async allreduce
t0 = time.perf_counter()
work = dist.all_reduce(tensor, op=dist.ReduceOp.SUM, async_op=True)

# Do other work here while collective is in flight
# e.g., update KV cache index, decode sampler bookkeeping
_ = torch.ones(256, 256, device=device).matmul(torch.ones(256, 256, device=device))

# Wait for collective to complete
work.wait()
torch.xpu.synchronize(device)
t1 = time.perf_counter()

if rank == 0:
    print(f"Async allreduce wall time (incl. overlap work): {(t1-t0)*1e6:.1f} us")
    print(f"Result[0] = {tensor[0].item():.1f}  (expected {dist.get_world_size()*(dist.get_world_size()+1)/2:.1f})")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allreduce_async.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 3. Latency Benchmark -- Message Size Sweep

This is the most important benchmark for TP inference tuning.
We sweep message sizes from 16KB (small batch decode) to 32MB (large batch prefill)
and measure allreduce latency.

**Expected shape on NUMA-only CRI hardware:**
- Small messages (< 256KB): latency-bound, relatively flat
- Large messages (> 1MB): bandwidth-bound, linear with size
- Crossover point tells you the effective PCIe bandwidth ceiling

In [ ]:
%%writefile /tmp/ccl_allreduce_bench.py
import os, time, json
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

WARMUP_ITERS = 10
BENCH_ITERS  = 50

# Message sizes in bytes (BF16 = 2 bytes/element)
sizes_bytes = [
    16   * 1024,    #  16 KB  -- TP=8 Llama-3 8B hidden=4096, batch=1
    32   * 1024,    #  32 KB
    128  * 1024,    # 128 KB  -- TP=4 Llama-3 70B hidden=8192, batch=1
    512  * 1024,    # 512 KB
    2    * 1024**2, #   2 MB  -- batch=16 decode
    8    * 1024**2, #   8 MB  -- prefill territory
    32   * 1024**2, #  32 MB  -- large prefill
]

results = []

for nbytes in sizes_bytes:
    nelems = nbytes // 2  # BF16
    tensor = torch.ones(nelems, dtype=torch.bfloat16, device=device)

    # Warmup
    for _ in range(WARMUP_ITERS):
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
        torch.xpu.synchronize(device)
    dist.barrier()

    # Benchmark
    latencies = []
    for _ in range(BENCH_ITERS):
        t0 = time.perf_counter()
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
        torch.xpu.synchronize(device)
        latencies.append((time.perf_counter() - t0) * 1e6)  # microseconds

    dist.barrier()

    latencies.sort()
    p50 = latencies[len(latencies)//2]
    p95 = latencies[int(len(latencies)*0.95)]

    if rank == 0:
        # Bus bandwidth: allreduce traffic = 2*(N-1)/N * msg_size
        bus_bw = 2 * (world_size-1) / world_size * nbytes / (p50 * 1e-6) / 1e9
        results.append({
            "size_kb": nbytes // 1024,
            "p50_us":  round(p50, 1),
            "p95_us":  round(p95, 1),
            "bus_gbps": round(bus_bw, 1)
        })

if rank == 0:
    print(f"\n{'Size':>10} {'p50 (us)':>12} {'p95 (us)':>12} {'BusBW (GB/s)':>14}")
    print("-" * 52)
    for r in results:
        print(f"{r['size_kb']:>8}KB  {r['p50_us']:>12.1f}  {r['p95_us']:>12.1f}  {r['bus_gbps']:>12.1f}")
    print()
    print(json.dumps(results))  # machine-readable for plotting

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allreduce_bench.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

In [ ]:
# Parse and plot the results
import json, re
import matplotlib.pyplot as plt
import numpy as np

# Extract JSON from output
json_line = [l for l in result.stdout.split('\n') if l.strip().startswith('[')]
if json_line:
    data = json.loads(json_line[-1])
    sizes  = [d['size_kb']  for d in data]
    p50    = [d['p50_us']   for d in data]
    p95    = [d['p95_us']   for d in data]
    busbw  = [d['bus_gbps'] for d in data]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.semilogx(sizes, p50, 'o-', color='#0071c5', label='p50', linewidth=2)
    ax1.semilogx(sizes, p95, 's--', color='#00aeef', label='p95', linewidth=1.5)
    ax1.set_xlabel('Message Size (KB)')
    ax1.set_ylabel('Latency (us)')
    ax1.set_title('Allreduce Latency vs Message Size\n(Ring, NUMA-only, BF16)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    # Mark typical TP decode region
    ax1.axvspan(16, 256, alpha=0.1, color='green', label='TP decode zone')
    ax1.text(50, max(p50)*0.7, 'TP decode\nregime', fontsize=8, color='green')

    ax2.semilogx(sizes, busbw, 'o-', color='#5b8f22', linewidth=2)
    ax2.set_xlabel('Message Size (KB)')
    ax2.set_ylabel('Bus Bandwidth (GB/s)')
    ax2.set_title('Effective Bus Bandwidth\n(theoretical ceiling = PCIe Gen5 ~64 GB/s)')
    ax2.axhline(y=64, linestyle='--', color='red', alpha=0.5, label='PCIe Gen5 ceiling')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle(f'oneCCL Ring Allreduce -- 4 Ranks, NUMA-Only CRI', fontweight='bold')
    plt.tight_layout()
    plt.savefig('/tmp/allreduce_bench.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Plot saved to /tmp/allreduce_bench.png")
else:
    print("No JSON output found -- run the benchmark cell first")

## 4. TP Decode Loop Simulation

This simulates what actually happens during autoregressive decode:
- 80 transformer layers (Llama-3 70B)
- 2 allreduces per layer (attn output projection + FFN down projection)
- Batch size 1 (pure latency mode)
- Measures per-token communication overhead

The output directly tells you **what fraction of TPOT is spent in collectives**.

:::{warning}
The cells below set `CCL_ALLREDUCE=ring` explicitly to isolate and benchmark the ring
algorithm on the CPU path. For **production GPU inference**, leave `CCL_ALLREDUCE` unset
so oneCCL uses the `topo` algorithm (GPU-native scale-up). Setting any value other than
`topo` causes oneCCL to copy GPU buffers to host memory and run a CPU algorithm.
Use `CCL_ALLREDUCE_SCALEOUT=ring` to control only the scaleout phase.
:::

In [ ]:
%%writefile /tmp/ccl_tp_decode_sim.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank   = dist.get_rank()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Model config: Llama-3 70B, TP=4
NUM_LAYERS   = 80    # Llama-3 70B
HIDDEN_DIM   = 8192
TP_DEGREE    = dist.get_world_size()
BATCH_SIZE   = 1
SEQ_LEN      = 1     # decode: one token at a time
NUM_TOKENS   = 100   # simulate 100 output tokens
WARMUP_TOKS  = 10

# Each allreduce message: (batch, seq, hidden) in BF16
# After TP sharding, each GPU holds hidden/TP slice, but allreduce
# operates on the full hidden dim output (not sharded output)
msg_elems = BATCH_SIZE * SEQ_LEN * HIDDEN_DIM
msg_bytes = msg_elems * 2  # BF16

if rank == 0:
    print(f"Config: Llama-3 70B | TP={TP_DEGREE} | batch={BATCH_SIZE} | hidden={HIDDEN_DIM}")
    print(f"Allreduce message: {msg_bytes/1024:.1f} KB ({msg_elems} BF16 elements)")
    print(f"Allreduces per token: {NUM_LAYERS * 2} (2 per layer x {NUM_LAYERS} layers)")
    print()

# Pre-allocate tensors (no allocation overhead in hot path)
attn_out  = torch.randn(msg_elems, dtype=torch.bfloat16, device=device)
ffn_down  = torch.randn(msg_elems, dtype=torch.bfloat16, device=device)

token_times = []

for tok in range(NUM_TOKENS + WARMUP_TOKS):
    dist.barrier()
    t_start = time.perf_counter()

    for layer in range(NUM_LAYERS):
        # Attention output projection (row-parallel → allreduce)
        dist.all_reduce(attn_out, op=dist.ReduceOp.SUM)
        # FFN down projection (row-parallel → allreduce)
        dist.all_reduce(ffn_down, op=dist.ReduceOp.SUM)

    torch.xpu.synchronize(device)
    t_end = time.perf_counter()

    if tok >= WARMUP_TOKS:
        token_times.append((t_end - t_start) * 1e3)  # ms

if rank == 0:
    token_times.sort()
    total_ar   = NUM_LAYERS * 2
    p50_tok    = token_times[len(token_times)//2]
    p95_tok    = token_times[int(len(token_times)*0.95)]
    p50_per_ar = p50_tok / total_ar * 1000  # us per allreduce

    print(f"--- TP Decode Allreduce Cost (comm only, no compute) ---")
    print(f"Total allreduces per token : {total_ar}")
    print(f"p50 total comm / token     : {p50_tok:.2f} ms")
    print(f"p95 total comm / token     : {p95_tok:.2f} ms")
    print(f"p50 per allreduce          : {p50_per_ar:.1f} us")
    print()
    print(f"If compute/token = 10 ms: comm overhead = {p50_tok/(10+p50_tok)*100:.1f}% of TPOT")
    print(f"If compute/token = 5  ms: comm overhead = {p50_tok/(5 +p50_tok)*100:.1f}% of TPOT")
    print(f"If compute/token = 2  ms: comm overhead = {p50_tok/(2 +p50_tok)*100:.1f}% of TPOT")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_tp_decode_sim.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 5. Environment Tuning -- Side-by-Side Comparison

This cell runs the same allreduce benchmark under different `CCL_*` settings
so you can quantify the impact of each tuning knob on your specific hardware.

In [ ]:
%%writefile /tmp/ccl_tuning_sweep.py
import os, time, sys
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

# Config passed via env by parent launcher
label = os.environ.get("BENCH_LABEL", "default")

dist.init_process_group(backend="ccl")
rank   = dist.get_rank()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# 128KB: typical TP=4 decode message
tensor = torch.ones(128 * 1024 // 2, dtype=torch.bfloat16, device=device)

for _ in range(20):  # warmup
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    torch.xpu.synchronize(device)

times = []
for _ in range(100):
    t0 = time.perf_counter()
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    torch.xpu.synchronize(device)
    times.append((time.perf_counter() - t0) * 1e6)

times.sort()
if rank == 0:
    print(f"{label:<40} p50={times[50]:.1f}us  p95={times[95]:.1f}us")

dist.destroy_process_group()

In [ ]:
import subprocess

configs = [
    {"BENCH_LABEL": "transport=ofi, workers=1, algo=ring",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_WORKER_COUNT": "1", "CCL_ALLREDUCE": "ring"},
    {"BENCH_LABEL": "transport=mpi, workers=1, algo=ring",
     "CCL_ATL_TRANSPORT": "mpi", "CCL_WORKER_COUNT": "1", "CCL_ALLREDUCE": "ring"},
    {"BENCH_LABEL": "transport=ofi, workers=2, algo=ring (not recommended for GPU)",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_WORKER_COUNT": "2", "CCL_ALLREDUCE": "ring"},  # Intel docs: keep <=1 for GPU buffers
    {"BENCH_LABEL": "transport=ofi, workers=1, algo=default",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_WORKER_COUNT": "1"},  # let CCL choose
]

print(f"{'Config':<40} {'p50':>10} {'p95':>10}")
print("-" * 65)

for cfg in configs:
    env = {**os.environ, "CCL_LOG_LEVEL": "error", **cfg}
    result = subprocess.run(
        "mpirun -n 4 -ppn 4 python /tmp/ccl_tuning_sweep.py",
        shell=True, capture_output=True, text=True, env=env
    )
    # Print rank-0 output (already formatted)
    for line in result.stdout.strip().split('\n'):
        if line.strip():
            print(line)

## Summary & Key Takeaways

| Takeaway | Detail |
|---|---|
| **Ring is correct for NUMA-only** | One-Shot requires direct GPU fabric; without XeLink/UALink it serializes on PCIe |
| **OFI transport beats MPI transport** | Lower overhead, no PMI roundtrip per collective |
| **1 CCL worker for decode** | Decode is latency-bound and sequential; extra workers waste cores |
| **Pre-allocate tensors** | Allocation in the decode hot path adds variance; reuse buffers |
| **Allreduce cost compounds** | 160 allreduces/token at 50us each = 8ms/token comm overhead -- significant at low compute budgets |
| **NIXL owns KV transport** | Do not route KV cache transfers through oneCCL; that belongs to NIXL/UCX layer |

**Next:** [Allgather for Sequence Parallelism](03b_allgather.ipynb)